# Assignment 2, notebook 2: Target, features and data preparation (Part B)
**Target:** bad = charge-off amount above zero (a sensitivity version counts only charge-offs within 60 months of approval).


**Split by time:** train on FY2014-2017, tune on FY2018, test on FY2019.


**Rule:** every feature must be known at the moment of approval.
Outputs: three model-ready tables (train / valid / test) in data/processed, plus the feature groups and settings used.

In [1]:
import json
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
TEAL, GREY, RED = "#0E7C86", "#B8BFCC", "#C0392B"
PROC = Path("../../data/processed"); OUT = Path("../outputs"); OUT.mkdir(parents=True, exist_ok=True)

In [2]:
# Target and time split

df = pd.read_parquet(PROC / "loans_clean.parquet")
d = df[df["approvalfy"].between(2014, 2019)].copy().reset_index(drop=True)
d["bad"] = (d["grosschargeoffamount"].fillna(0) > 0).astype(int)
d["months_to_co"] = (d["chargeoffdate"] - d["approvaldate"]).dt.days / 30.44
d["bad_60m"] = ((d["bad"] == 1) & (d["months_to_co"] <= 60)).astype(int)   # same 60-month window for every year
d["split"] = np.select([d["approvalfy"] <= 2017, d["approvalfy"] == 2018], ["train", "valid"], default="test")

d.groupby("split").agg(loans=("bad", "size"), first_fy=("approvalfy", "min"), last_fy=("approvalfy", "max"),
                       bad_rate=("bad", "mean"), bad_60m_rate=("bad_60m", "mean")).round(4)

,loans,first_fy,last_fy,bad_rate,bad_60m_rate
split,,,,,
test,45657,2019,2019,0.0659,0.0416
train,214087,2014,2017,0.0774,0.0442
valid,54172,2018,2018,0.0814,0.0500


In [3]:
# Two versions of the target
# Why: “ever charged off” falls in the newest year partly because those loans are younger. The 60-month version gives every year the same window, so it shows whether that fall is real.

yrs = d.groupby("approvalfy")[["bad", "bad_60m"]].mean().mul(100).reset_index().melt("approvalfy", var_name="target", value_name="rate")
yrs["target"] = yrs["target"].map({"bad": "Ever charged off", "bad_60m": "Charged off within 60 months"})
fig = px.bar(yrs, x="approvalfy", y="rate", color="target", barmode="group", text=yrs["rate"].round(1),
             color_discrete_sequence=[TEAL, GREY], template="plotly_white", title="Two versions of the target, by approval year (%)")
fig.add_vrect(x0=2013.5, x1=2017.5, fillcolor=GREY, opacity=0.12, line_width=0)   # shaded = training years
fig.show()



In [4]:
# Leakage audit
# Why: these columns describe what happened after approval, so a model that used them would be given the answer. We list every excluded column and the reason, so a reviewer can check.

LEAK = {"loanstatus": "outcome", "paidinfulldate": "outcome", "chargeoffdate": "outcome",
        "grosschargeoffamount": "outcome (the target)", "firstdisbursementdate": "happens after approval",
        "soldsecmrktind": "secondary-market sale happens after approval (and 71% blank)",
        "asofdate": "snapshot date (constant)", "approvalfy": "used only to split by time",
        "approvaldate": "used only to split by time", "months_to_co": "derived from outcome",
        "charged_off_ever": "outcome (same as target)"}
ID = ["borrname", "borrstreet", "borrcity", "borrzip", "bankstreet", "bankcity", "bankzip", "locationid",
      "program", "bankfdicnumber", "bankncuanumber", "franchisename"]
audit = pd.DataFrame({"column": list(LEAK) + ID,
                      "reason": list(LEAK.values()) + ["identifier / free text / constant (not a risk driver)"] * len(ID)})
audit[audit["column"].isin(d.columns)]

,column,reason
0,loanstatus,outcome
1,paidinfulldate,outcome
2,chargeoffdate,outcome
3,grosschargeoffamount,outcome (the target)
4,firstdisbursementdate,happens after approval
5,soldsecmrktind,secondary-market sale happens after approval (...
6,asofdate,snapshot date (constant)
7,approvalfy,used only to split by time
8,approvaldate,used only to split by time
9,months_to_co,derived from outcome


In [5]:
# Features from the loan and the borrower
# Why: these are the signals EDA showed matter, all known at approval. We use logs for skewed amounts, flags for the patterns we saw (short term, collateral, variable rate), 
# and mark invalid values (term 0, rate 0) as missing instead of trusting them.

tr = d["split"] == "train"
AGE = {"Startup, Loan Funds will Open Business": "Startup", "New, Less than 1 Year old": "Under 2y",
       "New Business or 2 years or less": "Under 2y", "Less than 3 years old but at least 2": "2-3y",
       "Less than 4 years old but at least 3": "3-4y", "Less than 5 years old but at least 4": "4-5y",
       "Existing or more than 2 years old": "2y+ (unspecified)", "Existing, 5 or more years": "5y+",
       "Change of Ownership": "Change of ownership"}

X = pd.DataFrame(index=d.index)
X["log_amount"] = np.log10(d["grossapproval"])
X["term_years"] = d["terminmonths"].where(d["terminmonths"] > 0) / 12
X["short_term"] = (d["terminmonths"] <= 60).astype(int)
X["rate"] = d["initialinterestrate"].where(d["initialinterestrate"] > 0)
X["variable_rate"] = (d["fixedorvariableinterestind"] == "V").astype(int)
X["rate_vs_year"] = X["rate"] - X.groupby(d["approvalfy"])["rate"].transform("median")   # rate relative to that year's market
X["guarantee_pct"] = d["guarantee_pct"]
X["jobs_log"] = np.log1p(d["jobssupported"])
X["no_jobs"] = (d["jobssupported"] == 0).astype(int)
X["amount_per_job_log"] = np.log10(d["grossapproval"] / d["jobssupported"].clip(lower=1))
X["has_collateral"] = d["collateralind"].astype(int)
X["revolving"] = d["revolverstatus"].astype(int)
X["franchise"] = d["franchisecode"].notna().astype(int)
X["out_of_state_lender"] = (d["borrstate"] != d["bankstate"]).astype(int)
X["term_missing"] = (d["terminmonths"] == 0).astype(int)
X["rate_missing"] = (d["initialinterestrate"] == 0).astype(int)
X.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
log_amount,313916.0,5.12,0.67,3.00,4.65,5.12,5.57,6.70
term_years,313788.0,10.30,6.63,0.08,7.00,8.00,10.00,36.00
short_term,313916.0,0.18,0.39,0.00,0.00,0.00,0.00,1.00
rate,313909.0,6.69,1.66,0.75,5.50,6.25,7.65,13.50
variable_rate,313916.0,0.77,0.42,0.00,1.00,1.00,1.00,1.00
rate_vs_year,313909.0,0.26,1.53,-6.45,-0.75,0.00,0.75,5.75
guarantee_pct,313916.0,0.64,0.14,0.10,0.50,0.75,0.75,0.90
jobs_log,313916.0,1.68,1.17,0.00,0.69,1.61,2.48,5.75
no_jobs,313916.0,0.18,0.38,0.00,0.00,0.00,0.00,1.00
amount_per_job_log,313916.0,4.47,0.63,1.92,4.04,4.43,4.88,6.70


In [6]:
# Categories, with rare levels grouped
# Why: small categories give noisy results and many empty columns. Any product, state or sector with under 1,000 training loans becomes “Other”. 
# “Unanswered” business age stays its own category, because not answering may carry information.

cats = pd.DataFrame(index=d.index)
cats["biz_age"] = d["businessage"].map(AGE).fillna("Unanswered")
cats["biz_type"] = d["businesstype"].fillna("Unknown")
cats["sector"] = d["naicscode"].astype("Int64").astype(str).str[:2]
cats["product"] = d["processingmethod"]
cats["state"] = d["borrstate"]

def top_levels(s, min_n=1000):
    vc = s[tr].value_counts()
    return set(vc[vc >= min_n].index)

for c in ["product", "state", "sector"]:
    cats[c] = cats[c].where(cats[c].isin(top_levels(cats[c])), "Other")
{c: cats[c].nunique() for c in cats}

{'biz_age': 9, 'biz_type': 4, 'sector': 20, 'product': 8, 'state': 44}

In [7]:
# # Encoding high-cardinality fields (industry and lender)
# Why: a 3-digit industry code has about 1,000 values and lenders about 2,000, too many to one-hot encode. 
# We replace each with its historical bad rate, smoothed towards the overall rate so small groups aren’t over-trusted. 
# For training rows we use out-of-fold values (each row’s rate is computed without itself), so the model can’t memorise its own answer. Valid and test rows use the training history only.

def smoothed_te(keys, y, train_mask, m=50, n_splits=5, seed=42):
    prior = y[train_mask].mean()
    out = pd.Series(prior, index=keys.index, dtype=float)
    def fit(k, t):
        g = pd.DataFrame({"k": k, "y": t}).groupby("k")["y"].agg(["sum", "count"])
        return (g["sum"] + m * prior) / (g["count"] + m)
    tr_idx = np.where(train_mask)[0]
    for a, b in KFold(n_splits, shuffle=True, random_state=seed).split(tr_idx):
        mp = fit(keys.iloc[tr_idx[a]], y.iloc[tr_idx[a]])
        out.iloc[tr_idx[b]] = keys.iloc[tr_idx[b]].map(mp).fillna(prior).values
    mp = fit(keys[train_mask], y[train_mask])
    out[~train_mask] = keys[~train_mask].map(mp).fillna(prior).values
    return out, mp

naics3 = d["naicscode"].astype("Int64").astype(str).str[:3]
X["naics3_te"], _ = smoothed_te(naics3, d["bad"], tr.values)
X["lender_te"], lender_map = smoothed_te(d["bankname"], d["bad"], tr.values)

lender_n = d.loc[tr, "bankname"].value_counts()
X["lender_log_volume"] = np.log10(d["bankname"].map(lender_n).fillna(0) + 1)
X["lender_seen"] = d["bankname"].isin(lender_n.index).astype(int)
sblc = set(pd.read_csv("../../assignment1/outputs/sblc_list.csv")["bankname"])
X["lender_is_sblc"] = d["bankname"].isin(sblc).astype(int)

print("Share of valid / test loans from lenders seen in training:",
      X.loc[d.split == "valid", "lender_seen"].mean().round(3), X.loc[d.split == "test", "lender_seen"].mean().round(3))

Share of valid / test loans from lenders seen in training: 0.996 0.993


In [8]:
# Missing values, outliers, one-hot encoding
# Why: medians and caps are learned on training data only and then applied to valid and test, so nothing leaks from the future. 
# Extremes are capped at the 0.5th and 99.5th percentiles so a few huge values don’t dominate. 
# The raw interest rate is dropped because rates rose over the years (it carries the calendar); rate_vs_year keeps the part that signals risk.

medians = X[tr].median()
X = X.fillna(medians)
caps = {c: (X.loc[tr, c].quantile(0.005), X.loc[tr, c].quantile(0.995))
        for c in ["jobs_log", "amount_per_job_log", "rate_vs_year", "log_amount"]}
for c, (lo, hi) in caps.items():
    X[c] = X[c].clip(lo, hi)

X = X.drop(columns="rate")
dm = pd.get_dummies(cats, dtype=int)
dm = dm.loc[:, dm[tr].sum() > 0]
M = pd.concat([X, dm], axis=1)
print("Model table:", M.shape, "| missing values:", int(M.isna().sum().sum()))

Model table: (313916, 104) | missing values: 0


In [9]:
# Class imbalance
# Why: about 1 bad loan in 13 is moderate imbalance. We keep the data as is and deal with it in modelling (class weights, and we test resampling), since resampling must only ever touch training data.

y = d["bad"]
pos = y[tr].mean()
print(f"Bad-loan share in training: {pos:.2%} | good:bad = {(1 - pos) / pos:.1f}:1 | class weight for bad loans: {(1 - pos) / pos:.1f}")

Bad-loan share in training: 7.74% | good:bad = 11.9:1 | class weight for bad loans: 11.9


In [10]:
# Leakage check and a first look at the signal
# Why: a single feature that predicts far too well is the classic sign of leakage. This scores each feature alone, with AUC (0.5 = no signal, 1.0 = perfect).

rows = []
for c in M.columns:
    if M.loc[tr, c].nunique() > 1:
        a = roc_auc_score(y[tr], M.loc[tr, c])
        rows.append((c, max(a, 1 - a), "up" if a > .5 else "down"))
uni = pd.DataFrame(rows, columns=["feature", "single_feature_auc", "higher_value_means"]).sort_values("single_feature_auc", ascending=False)

top = uni.head(15).iloc[::-1]
fig = px.bar(top, x="single_feature_auc", y="feature", orientation="h", color_discrete_sequence=[TEAL],
             template="plotly_white", title="Single-feature AUC on training data (0.5 = no signal)")
fig.add_vline(x=0.5, line_dash="dot")
fig.show()
uni.head(12).round(3)

,feature,single_feature_auc,higher_value_means
1,term_years,0.810,down
2,short_term,0.711,up
16,lender_te,0.667,up
4,rate_vs_year,0.659,up
0,log_amount,0.588,down
15,naics3_te,0.580,up
8,amount_per_job_log,0.574,down
17,lender_log_volume,0.570,up
9,has_collateral,0.562,down
23,biz_age_5y+,0.560,down


In [11]:
# Do the splits look alike?
# Why: if the loan mix changes a lot over time, the model faces a different population when it is used. This compares the averages of key features across the three splits.

M.assign(split=d["split"], rate=d["initialinterestrate"]).groupby("split")[
    ["log_amount", "term_years", "rate", "variable_rate", "has_collateral", "short_term", "lender_te"]].mean().round(3)

,log_amount,term_years,rate,variable_rate,has_collateral,short_term,lender_te
split,,,,,,,
test,5.194,10.965,7.909,0.791,0.740,0.140,0.085
train,5.098,10.062,6.275,0.770,0.729,0.197,0.081
valid,5.150,10.653,7.291,0.778,0.719,0.167,0.087


In [12]:
# Feature groups
# Why: we label features by type (loan, borrower, industry and place, product, lender). In notebook 3 we can then switch a group off and see how much the model depends on it. 
# The check at the end makes sure every column belongs to exactly one group.

GROUPS = {"loan": ["log_amount", "term_years", "short_term", "variable_rate", "rate_vs_year", "guarantee_pct",
                   "has_collateral", "revolving", "term_missing", "rate_missing"],
          "borrower": ["jobs_log", "no_jobs", "amount_per_job_log", "franchise"] + [c for c in M if c.startswith(("biz_age_", "biz_type_"))],
          "industry_geo": ["naics3_te", "out_of_state_lender"] + [c for c in M if c.startswith(("sector_", "state_"))],
          "product": [c for c in M if c.startswith("product_")],
          "lender": ["lender_te", "lender_log_volume", "lender_seen", "lender_is_sblc"]}
assert sorted(sum(GROUPS.values(), [])) == sorted(M.columns)
{k: len(v) for k, v in GROUPS.items()}

{'loan': 10, 'borrower': 16, 'industry_geo': 66, 'product': 8, 'lender': 4}

In [13]:
# save the file

meta = d[["approvalfy", "bankname", "grossapproval", "grosschargeoffamount"]]
full = pd.concat([M, d[["bad", "bad_60m", "split"]], meta], axis=1)
for name in ["train", "valid", "test"]:
    full[full["split"] == name].drop(columns="split").to_parquet(PROC / f"a2_{name}.parquet", index=False)

json.dump(GROUPS, open(OUT / "a2_feature_groups.json", "w"), indent=1)
json.dump({"train_medians": medians.round(6).to_dict(),
           "caps": {k: [float(x) for x in v] for k, v in caps.items()},
           "target": "bad = charge-off amount > 0", "split": "train FY2014-17, valid FY2018, test FY2019"},
          open(OUT / "a2_prep_config.json", "w"), indent=1)
pd.DataFrame([(f, g) for g, fs in GROUPS.items() for f in fs], columns=["feature", "group"]).merge(uni, on="feature", how="left") \
  .to_csv(OUT / "a2_feature_dictionary.csv", index=False)

chk = pd.read_parquet(PROC / "a2_test.parquet")
print(chk.shape, "bad rate:", chk["bad"].mean().round(4), "| missing in features:", int(chk[list(M.columns)].isna().sum().sum()))

(45657, 110) bad rate: 0.0659 | missing in features: 0


## Summary

**Target and split**
- Bad = charge-off amount above zero. Train FY2014-17 (214,087 loans, 7.7% bad), validate FY2018 (54,172, 8.1%), test FY2019 (45,657, 6.6%).
- On a fixed 60-month window the bad rate is steadier across years (about 4.2-5.0%), so part of FY2019's lower "ever" rate is simply because its loans are younger.

**Features (110 columns in 5 groups)**
- Loan (10), borrower (16), industry and place (66), product (8), lender (4).
- Everything is known at approval. Outcome fields, post-approval dates and the secondary-market flag were excluded (see the audit table).
- Invalid values (term 0, rate 0) became missing, then were filled with training medians and flagged. Extremes were capped. Rare categories were grouped. Industry and lender history were encoded out-of-fold.

**Checks**
- Strongest single feature is loan term (AUC 0.81), then short-term flag (0.71), lender history (0.67) and rate versus the year's market (0.66). Nothing is suspiciously close to 1.0, so there is no sign of leakage. Term is known at approval, so a high score is expected.
- The loan mix shifts over time. Average interest rates rose from 6.3% (train) to 7.9% (test) as market rates rose, which is why we use the rate relative to that year's market instead of the raw rate.
- 99% of valid and test loans come from lenders seen in training.

**Risks to carry forward**
- **Lender history is optimistic.** For a loan approved in FY2018, outcomes of FY2017 loans would not yet be known in real life (charge-offs take about 4.5 years). Our lender history uses all FY2014-17 outcomes. In notebook 3 we run the model with and without the lender group, and also test on lenders it has not seen.
- **Population shift** (rates, loan mix) means the FY2019 score is the honest test, not the random-split score.